In [ ]:
%matplotlib widget
import os
import sys
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torchkbnufft as tkbn
import pypulseq as pp
import MRzeroCore as mr0
import matplotlib.pyplot as plt
from tqdm import tqdm

def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for d in [cur, *cur.parents]:
        if (d / "Modules").is_dir():
            return d
    raise RuntimeError(f"Could not find a folder containing 'Modules' from {cur}")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Drop any cached Modules* modules so a kernel restart is only needed for
# *new* modules, not for edits to existing ones - re-running this cell
# always re-imports the current Modules/ code.
for _m in [m for m in list(sys.modules) if m == 'Modules' or m.startswith('Modules.')]:
    del sys.modules[_m]

from Modules import PARAMS, build_sim_data
from Modules.bssfp_params import TISSUES, T2STAR, NAMES
from Modules.recon_ops import ChannelOperator, recon_nufft, recon_cg, signed_series, mag_series
from Modules.Plan import KbNufft, KbNufftAdjoint
from Modules.NumSolvers import conjugate_gradient

torch.set_default_device("cuda")
device = torch.device("cuda")

In [ ]:
signal_name = "bssfp_kooshball3D_signal.pt"   # <-- choose signal
signal_file = REPO_ROOT / "Simulation & Visualization" / "Signals" / signal_name
if not signal_file.exists():
    raise FileNotFoundError(
        f"Missing .pt: {signal_file}\nRun Simulate.ipynb first to generate it.")
print("signal file:", signal_file)

In [ ]:
signal = torch.load(signal_file)

In [ ]:
seq_name = "bssfp_kooshball3D_signal_n64_R384_S25.seq"   # <-- choose sequence
seq_file = REPO_ROOT / "Sequences" / "Sequence files" / seq_name
if not seq_file.exists():
    raise FileNotFoundError(
        f"Missing .seq: {seq_file}\nRun 3D_bssfp_sequence.ipynb first to generate it.")
print("sequence file:", seq_file)

In [ ]:
mr_seq = mr0.Sequence.import_file(str(seq_file))   # <-- import the sequence
print("imported sequence, repetitions:", len(mr_seq))

In [ ]:
data, masks, brain_mask, coil_den, coil_c64 = build_sim_data(
    raw_path=str(REPO_ROOT / "Data" / "subject04_crisp_v.rawb"),  # <-- <repo>/Data/
    p=PARAMS,
    device=torch.device("cpu"),
    permute_order=(2, 1, 0),
)

coil_den = coil_den.to(device)
coil_c64 = coil_c64.to(device)
# Bind the grid / readout names the cells below already use (same values as the
# sequence notebook, straight from the shared protocol params):
n          = PARAMS.n
FOV        = PARAMS.FOV
N_x        = PARAMS.N_x
N_readout  = PARAMS.N_readout
N_segments = PARAMS.N_segments
N_channels = PARAMS.N_channels
n_adc      = PARAMS.n_adc
TR_echo    = PARAMS.TR_echo
TI_START   = PARAMS.TI_START
P_echo     = PARAMS.P_echo

In [ ]:
def kspace_to_rad(mr_seq, n, FOV):
    'ADC k-space (1/m) -> rad/pixel of an n^3 grid spanning FOV.  Returns (3, N_adc).'
    k = torch.as_tensor(mr_seq.get_kspace()).float()
    k = k.reshape(-1, k.shape[-1])[:, :3].T.contiguous()
    return k * (2 * np.pi * FOV / n)

k_full = kspace_to_rad(mr_seq, n, FOV)
assert k_full.shape[1] == signal.shape[1], (k_full.shape, signal.shape)
k_full = k_full.to(device)
keep_full = k_full.norm(dim=0) <= np.pi
k_norm = k_full / k_full.norm(dim=0).max() * 0.5     # cycles/voxel, for the CG operator

In [ ]:
FIB = [8, 13, 21, 34, 55, 89, 144, 233, 377, 610]
def _next_fib(x):
    return next(f for f in FIB if f >= x)

N_center = 8      # echoes for the k-space center
N_max    = 55     # outermost Fibonacci ring (= time window at the periphery)
frame_advance = 4 # echo-index advance per frame (8 * 12.65 ms = ~101 ms)

rad = k_full.reshape(3, N_segments, N_readout, N_x)[:, 0, 0, :].norm(dim=0)
r_frac = rad / rad.max()
ring = np.array([_next_fib(max(N_center, int(np.ceil(N_max * float(r))))) for r in r_frac])
print(f"ring: center={ring.min()}, edge={ring.max()}, sum={ring.sum()}")

# butterfly offsets: for radial sample jj, the ring[jj] echo indices centered on 0
djs, iis = [], []
for jj in range(N_x):
    lo = -(int(ring[jj]) // 2)
    djs.extend(range(lo, lo + int(ring[jj])))
    iis.extend([jj] * int(ring[jj]))
dj = np.asarray(djs, dtype=np.int64)     # (Kb,) echo-index offset
ii = np.asarray(iis, dtype=np.int64)     # (Kb,) radial sample index
Kb = dj.size

# frame centers (echo index), windows clipped at the boundaries
centers = list(range(0, N_readout, frame_advance))          # 0, 8, ..., 376
n_frames = len(centers)
ti_frames = np.array([TI_START + c * P_echo for c in centers])
print(f"frames: {n_frames}, TI span {ti_frames[0]*1e3:.0f} - {ti_frames[-1]*1e3:.0f} ms")

DEPHASE = False
seg_off = (np.arange(N_segments)[:, None] * (N_readout * N_x))  # (S, 1)
keep_np = keep_full.cpu().numpy()
kwic_idx, kwic_w = [], []
for c in centers:
    e = np.clip(c + dj, 0, N_readout - 1)                  # (Kb,) echo index
    flat = (seg_off + (e[None, :] * N_x + ii[None, :])).ravel()   # (S*Kb,)
    flat = flat[np.asarray(keep_np[flat], dtype=bool)]     # drop |k| > pi
    if DEPHASE:
        w = np.where(((flat // N_x) % N_readout) % 2 == 0, 1.0, -1.0).astype(np.float32)
    else:
        w = np.ones(len(flat), dtype=np.float32)
    kwic_idx.append(torch.as_tensor(flat, device=device))
    kwic_w.append(torch.as_tensor(w, device=device))
K_frame = kwic_idx[0].numel()
print(f"K per frame: {K_frame} (={N_segments} x {Kb} per segment)")

# k-space for the CG operator (cycles/voxel)
kwic_k = [k_norm[:, idx] for idx in kwic_idx]

In [ ]:
C_op = ChannelOperator(coil_c64, (n, n, n))

import Modules.recon_ops as ro
ro.__dict__.update(globals())

t1 = time.time()
frames_nufft = {}
for f in range(n_frames):
    frames_nufft[f] = recon_nufft(f)
    #print(f"frame {f:2d}/{n_frames-1}  TI = {ti_frames[f]*1e3:6.0f} ms   (NUFFT {time.time()-t1:.0f} s)")
print(f"all NUFFT previews: {time.time()-t1:.0f} s")

In [ ]:
RUN_CG = True
CG_ITERS = 40

CG_FRAMES = list(range(n_frames - 75))

C_op = ChannelOperator(coil_c64, (n, n, n))
import Modules.recon_ops as ro
ro.__dict__.update(globals())

if RUN_CG:
    t1 = time.time()
    frames_cg = {}
    for f in tqdm(CG_FRAMES, desc="CG-SENSE"):
        frames_cg[f] = recon_cg(f, maxiter=CG_ITERS)
        if f % 8 == 0:
            print(f"  frame {f:2d}  TI = {ti_frames[f]*1e3:6.0f} ms   ({time.time()-t1:.0f} s)")
    print(f"all CG frames ({len(CG_FRAMES)}): {time.time()-t1:.0f} s")
    frames_fit = {f: frames_cg[f] for f in CG_FRAMES}
else:
    frames_fit = {f: frames_nufft[f] for f in CG_FRAMES}

print(f"using {'CG' if RUN_CG else 'NUFFT'} reconstructions for quantification ({len(CG_FRAMES)} frames)")

In [ ]:
mid = n // 2
fit_frames = sorted(frames_fit.keys())
grid_frames = fit_frames[::max(1, len(fit_frames) // 24)][:24]
_slic = np.stack([frames_fit[f].abs().cpu().numpy()[:, :, mid] for f in grid_frames])
#vmax_shared = float(np.percentile(_slic, 95))
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
for i, f in enumerate(grid_frames):
    ax = axes.ravel()[i]
    img = frames_fit[f].abs().cpu().numpy()
    ax.imshow(np.rot90(img[:, :, mid].T, 2), cmap="gray")
    ax.set_title(f"TI = {ti_frames[f]*1e3:4.0f} ms", fontsize=8)
    ax.axis("off")
for j in range(len(grid_frames), axes.size):
    axes.ravel()[j].axis("off")
plt.suptitle("kwic bSSFP - one scan, one image per echo-index window (magnitude)")
plt.tight_layout()
plt.show()

In [ ]:
half_win = N_max // 2
# phase reference: the last COMPUTED frame whose window is not clipped
ref_f = max(f for f in fit_frames if centers[f] + half_win < N_readout)
ref_phase = torch.angle(frames_fit[ref_f])
t_fit = np.array([ti_frames[f] for f in fit_frames])
print(f"phase reference: frame {ref_f} (TI = {ti_frames[ref_f]*1e3:.0f} ms)")

import Modules.recon_ops as ro
ro.__dict__.update(globals())

fig, axes = plt.subplots(1, 2, figsize=(8, 4.5))
for t, col in zip(NAMES, ["tab:blue", "tab:orange", "tab:green"]):
    axes[0].plot(t_fit * 1e3, signed_series(masks[t]), label=t)
    axes[1].plot(t_fit * 1e3, mag_series(masks[t]), label=t)
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_title("signed tissue-mean IR transient (SS-phase referenced)")
axes[1].set_title("magnitude tissue-mean (unsigned)")
for a in axes:
    a.set_xlabel("TI (ms)"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout()
plt.show()